# Inferensi Time-Series + Output Kontrak WebGIS (Tahap 2-3)

Notebook ini mengambil komposit citra (image-only, 36 tile/tahun) yang sudah di-export ke
Google Drive (lihat `export_inference_years.ipynb`), menjalankan model 7-kelas, lalu menyusun
**semua file yang dibutuhkan WebGIS Kasuari AI** ke SATU folder Drive baru: **`Output_Fix`**.

**Alur:**
1. Load model (checkpoint `.pt`) sekali.
2. Inferensi tile-by-tile per tahun yang tersedia (`papua_{tahun}_tile_*.tif` -> mask lokal).
3. Mosaic mask -> `landcover_{tahun}.png` + `landcover_{tahun}_bounds.json` per tahun
   (Sekarang dibuat untuk **semua tahun** yang ada, bukan cuma 2021/2025 -- bonus utk
   time-series slider yang sudah ada di frontend).
4. Change detection **kumulatif 2021->2025** (T1 vs T2) -> `deforestation.geojson`
   (kontrak WebGIS cuma punya 1 file ini, jadi tak dibuat per-tahun-pasangan).
5. `statistics.json`, `legend.json`, `metrics.json`, `model_card.md`.
6. Export ONNX dari model yang BARU SAJA dipakai inferensi (bukan copy file lama yg mungkin
   beda checkpoint) -> `model.onnx`.

**Catatan jujur -- gap data (`per_province` di statistics.json):**
Deteksi perubahan butuh nama provinsi per polygon untuk chart "Peringkat Provinsi" di
Dashboard. **Tidak ada dataset batas 6 provinsi Papua (pasca pemekaran 2022) di repo ini** --
sudah dicek: `dummy.py` cuma random-pilih provinsi, tak ada shapefile/GeoJSON batas asli.
Notebook ini punya slot opsional (`PROVINCE_BOUNDARY_GEOJSON`, Bagian 4) -- isi kalau sudah
punya file batas provinsi; kalau dikosongkan (default), polygon tetap dibuat tapi TANPA tag
provinsi (chart per-provinsi akan kosong utk data asli, tapi semua data lain tetap lengkap).


## Bagian 0 -- Setup environment (Colab) + mount Drive

Tak perlu auth GEE di notebook ini (cuma baca GeoTIFF tile dari Drive, tak ada panggilan GEE).


In [ ]:
# === Bagian 0 -- Setup environment (Colab) + mount Drive ===
import os, sys, subprocess
from pathlib import Path

subprocess.run("cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
               "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
               shell=True, check=False)
subprocess.run('pip install -q -e "/content/fw_repo/model[ml,gis]"', shell=True, check=False)

_SRC = "/content/fw_repo/model/src"
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

from google.colab import drive  # noqa: PLC0415
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
print("Drive ter-mount:", DRIVE_ROOT.exists())


## Bagian 1 -- Config

Set tahun yang mau diproses, lokasi tile di Drive, checkpoint model, dan folder output baru
(`Output_Fix`). Tahun yang foldernya belum ada di Drive otomatis dilewati (bukan error) --
jalankan ulang notebook ini kapan saja tahun baru selesai export.


In [ ]:
# === Bagian 1 -- Config ===
YEARS = [2021, 2022, 2023, 2024, 2025]              # tahun yg DICOBA diproses (skip kalau blm ada)
T1_YEAR, T2_YEAR = 2021, 2025                       # pasangan resmi utk deforestation.geojson

# Semua aset proyek (tile, training, checkpoint) hidup di SATU folder ini di Drive kamu --
# dicek langsung dari Drive: "Drive Saya > Satria Data 3.0 > ...".
PROJECT_ROOT = DRIVE_ROOT / "Satria Data 3.0"

TILE_DIR_TMPL = "ForestWatch_Tiles_{year}"          # folder DI BAWAH PROJECT_ROOT, image-only
TILE_GLOB_TMPL = "papua_{year}_tile_*.tif"

# 2021 (T1) & 2025 (T2) BUKAN folder per-tahun -- itu export awal (full pipeline + label),
# namanya tetap "ForestWatch_Tiles_T1"/"_T2" (dicek dari Drive). Naming file di dalamnya
# belum diverifikasi, jadi glob dibuat permisif ("*.tif") utk folder ini saja.
TILE_DIR_OVERRIDE = {T1_YEAR: "ForestWatch_Tiles_T1", T2_YEAR: "ForestWatch_Tiles_T2"}
TILE_GLOB_OVERRIDE = {T1_YEAR: "*.tif", T2_YEAR: "*.tif"}

OUT_DIR = PROJECT_ROOT / "Output_Fix"               # <- folder baru, SEMUA output webgis di sini
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_MASK_DIR = Path("/content/masks")             # mask hasil inferensi -- lokal (sementara, bukan Drive)
LOCAL_MASK_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint + metrik TEST -- dicek langsung dari Drive:
# Satria Data 3.0/Bahan_Training_Fix_Combined_v4/output/{best_model_finetune_v2.pt,metrics_finetune.json}
TRAINING_OUTPUT_DIR = PROJECT_ROOT / "Bahan_Training_Fix_Combined_v4" / "output"
MODEL_CHECKPOINT_PATH = TRAINING_OUTPUT_DIR / "best_model_finetune_v2.pt"
MODEL_ARCH = dict(architecture="unet_scse", encoder_name="resnet50", encoder_weights=None)
IN_CHANNELS, PATCH_SIZE, STRIDE, TTA = 6, 256, 128, False

# Jumlah sliding-window digabung per forward-pass GPU (bukan "worker"/multiprocessing --
# cuma ada 1 GPU, proses paralel antar-tile akan rebutan GPU yg sama, bukan mempercepat).
# Naikkan kalau VRAM masih longgar (mis. 32), turunkan kalau OOM (mis. 8).
INFER_BATCH_SIZE = 16

# Metrik TEST hasil training -- file ini SUDAH ADA (terlihat di Drive), jadi statistics.json/
# metrics.json akan pakai angka asli, bukan kosong/dikarang.
METRICS_SRC_JSON = TRAINING_OUTPUT_DIR / "metrics_finetune.json"

# Opsional: GeoJSON batas 6 provinsi Papua (utk tag `province` di tiap polygon perubahan).
# Kosongkan (None) kalau belum punya -- lihat catatan gap data di markdown atas.
PROVINCE_BOUNDARY_GEOJSON = None
PROVINCE_NAME_FIELD = "province"  # nama field di properties GeoJSON boundary yg berisi nama provinsi

MIN_AREA_HA = 0.5
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

print("PROJECT_ROOT   :", PROJECT_ROOT, "| ada:", PROJECT_ROOT.exists())
print("OUT_DIR        :", OUT_DIR)
print("MODEL_CHECKPOINT_PATH:", MODEL_CHECKPOINT_PATH, "| ada:", MODEL_CHECKPOINT_PATH.exists())
print("METRICS_SRC_JSON     :", METRICS_SRC_JSON, "| ada:", METRICS_SRC_JSON.exists())
print("DEVICE         :", DEVICE, "-- GANTI ke GPU (Runtime > Change runtime type) kalau masih 'cpu'.")
print("INFER_BATCH_SIZE:", INFER_BATCH_SIZE)
print("Province join  :", "AKTIF" if PROVINCE_BOUNDARY_GEOJSON else "DILEWATI (gap data, lihat catatan)")


## Bagian 2 -- Load model (sekali, dipakai utk semua tahun + export ONNX)

In [ ]:
# === Bagian 2 -- Load model ===
import torch
from forestwatch.model.architecture import build_unet
from forestwatch.constants import N_CLASSES

assert MODEL_CHECKPOINT_PATH.exists(), f"Checkpoint tak ditemukan: {MODEL_CHECKPOINT_PATH}"

model = build_unet(in_channels=IN_CHANNELS, classes=N_CLASSES, **MODEL_ARCH).to(DEVICE)
model.load_state_dict(torch.load(MODEL_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model siap ({n_params:,} parameter) di {DEVICE}. Checkpoint: {MODEL_CHECKPOINT_PATH.name}")


## Bagian 3 -- Inferensi per tahun (tile -> mask lokal)

Tahun yang folder Drive-nya belum ada / kosong otomatis dilewati (di-print sbg PERINGATAN,
bukan error) -- supaya notebook tetap bisa dijalankan walau export tahun lain masih berjalan.


In [ ]:
# === Bagian 3 -- Inferensi per tahun ===
from forestwatch.inference.tile_inference import infer_tiles_folder

mask_dirs = {}   # {year: Path lokal berisi mask_*.tif}
available_years = []

for year in YEARS:
    tile_dir = PROJECT_ROOT / TILE_DIR_OVERRIDE.get(year, TILE_DIR_TMPL.format(year=year))
    tile_glob = TILE_GLOB_OVERRIDE.get(year, TILE_GLOB_TMPL.format(year=year))
    n_tiles = len(list(tile_dir.glob(tile_glob))) if tile_dir.exists() else 0
    if n_tiles == 0:
        print(f"[SKIP] {year}: folder '{tile_dir.name}' tak ada / kosong -- export blm selesai.")
        continue

    out_dir = LOCAL_MASK_DIR / str(year)
    print(f"[{year}] inferensi {n_tiles} tile (batch={INFER_BATCH_SIZE}) -> {out_dir}")
    infer_tiles_folder(
        tile_dir, out_dir, model,
        prefix="mask_", tile_glob=tile_glob,
        device=DEVICE, patch_size=PATCH_SIZE, stride=STRIDE,
        n_channels_image=IN_CHANNELS, tta=TTA, batch_size=INFER_BATCH_SIZE,
    )
    mask_dirs[year] = out_dir
    available_years.append(year)

print(f"\nTahun siap (ada mask): {available_years}")
assert T1_YEAR in available_years and T2_YEAR in available_years, (
    f"Pasangan resmi T1={T1_YEAR}/T2={T2_YEAR} butuh KEDUANYA tersedia utk deforestation.geojson."
)


## Bagian 4 -- Render landcover PNG (semua tahun tersedia) + deforestation.geojson

In [ ]:
# === Bagian 4a -- landcover_{tahun}.png + bounds (semua tahun yg ada mask) ===
from forestwatch.outputs.landcover_png import mosaic_masks_to_png

landcover_paths = {}
for year in available_years:
    mask_files = sorted(mask_dirs[year].glob("mask_*.tif"))
    png_path, bounds_path = mosaic_masks_to_png(
        mask_files,
        OUT_DIR / f"landcover_{year}.png",
        OUT_DIR / f"landcover_{year}_bounds.json",
    )
    landcover_paths[year] = (png_path, bounds_path)
    print(f"[{year}] {png_path.name} + {bounds_path.name}")


In [ ]:
# === Bagian 4b -- deforestation.geojson (kumulatif T1->T2, kontrak WebGIS) ===
from forestwatch.inference.change_detection import detect_transitions

fc = detect_transitions(
    t1_dir=mask_dirs[T1_YEAR], t2_dir=mask_dirs[T2_YEAR],
    out_geojson=OUT_DIR / "deforestation.geojson",
    t1_prefix="mask_", t2_prefix="mask_",
    period_from=T1_YEAR, period_to=T2_YEAR, min_area_ha=MIN_AREA_HA,
)
print(f"deforestation.geojson: {len(fc['features'])} polygon perubahan ({T1_YEAR}->{T2_YEAR}).")

# --- Province join opsional (lihat config) ---
if PROVINCE_BOUNDARY_GEOJSON is not None:
    from shapely.geometry import shape
    from forestwatch.utils.io import load_json, save_geojson

    boundary_fc = load_json(PROVINCE_BOUNDARY_GEOJSON)
    boundary_polys = [
        (shape(f["geometry"]), f["properties"].get(PROVINCE_NAME_FIELD, "?"))
        for f in boundary_fc["features"]
    ]
    n_tagged = 0
    for feat in fc["features"]:
        centroid = shape(feat["geometry"]).centroid
        for poly, name in boundary_polys:
            if poly.contains(centroid):
                feat["properties"]["province"] = name
                n_tagged += 1
                break
    save_geojson(fc, OUT_DIR / "deforestation.geojson")
    print(f"Province join: {n_tagged}/{len(fc['features'])} polygon ter-tag provinsi.")
else:
    print("Province join DILEWATI (PROVINCE_BOUNDARY_GEOJSON=None) -- polygon tanpa tag provinsi.")


## Bagian 5 -- statistics.json, legend.json, metrics.json, model_card.md

In [ ]:
# === Bagian 5 -- statistics.json + legend.json + metrics.json + model_card.md ===
from forestwatch.outputs.landcover_png import compute_per_class_area_ha_from_geotiff
from forestwatch.outputs.legend import build_legend_json
from forestwatch.outputs.model_card import render_model_card
from forestwatch.outputs.statistics import build_statistics_json, save_metrics_json
from forestwatch.utils.io import load_json

# Per-class area dari mask T2 (kondisi tutupan lahan terbaru)
from collections import defaultdict
per_class_area = defaultdict(float)
for f in sorted(mask_dirs[T2_YEAR].glob("mask_*.tif")):
    for name, ha in compute_per_class_area_ha_from_geotiff(f).items():
        per_class_area[name] += ha
per_class_area = {k: round(v, 1) for k, v in per_class_area.items()}

model_metrics = load_json(METRICS_SRC_JSON) if METRICS_SRC_JSON is not None else None
if model_metrics is None:
    print("PERINGATAN: METRICS_SRC_JSON=None -> model_metrics di statistics.json/metrics.json "
          "akan KOSONG (0.0), bukan dikarang. Isi config kalau file metrik training sudah ada.")

build_statistics_json(
    period_from=T1_YEAR, period_to=T2_YEAR,
    deforestation_geojson=fc, per_class_area_ha=per_class_area,
    model_metrics=model_metrics, out_path=OUT_DIR / "statistics.json",
)
build_legend_json(out_path=OUT_DIR / "legend.json")
if model_metrics is not None:
    save_metrics_json(model_metrics, OUT_DIR / "metrics.json")
render_model_card(
    OUT_DIR / "model_card.md",
    n_parameters=n_params, metrics=model_metrics or {},
)
print("statistics.json, legend.json, model_card.md tersimpan ke", OUT_DIR)


## Bagian 6 -- Export ONNX dari model yang BARU dipakai inferensi

Diekspor langsung dari `model` di memori (bukan copy file `.onnx` lama) -- menjamin file ONNX
ini PERSIS sama dengan model yang menghasilkan mask/PNG/geojson di atas.


In [ ]:
# === Bagian 6 -- Export ONNX ke Output_Fix ===
from forestwatch.model.architecture import export_to_onnx

onnx_path = export_to_onnx(
    model, OUT_DIR / "model.onnx",
    in_channels=IN_CHANNELS, patch_size=PATCH_SIZE,
)
print("ONNX disimpan:", onnx_path)


## Bagian 7 -- Ringkasan & cross-check vs kontrak WebGIS

WebGIS (`webgis/backend/app/core/config.py`) butuh persis file-file ini di `Output_Fix`
(landcover per tahun bersifat tambahan/bonus -- backend saat ini baru baca 2021 & 2025).


In [ ]:
# === Bagian 7 -- Ringkasan isi Output_Fix ===
REQUIRED_BY_WEBGIS = [
    "landcover_2021.png", "landcover_2021_bounds.json",
    "landcover_2025.png", "landcover_2025_bounds.json",
    "deforestation.geojson", "statistics.json", "legend.json", "model.onnx",
]
present = {p.name for p in OUT_DIR.iterdir()}
print(f"Isi {OUT_DIR}:")
for name in sorted(present):
    print(" -", name)

missing = [f for f in REQUIRED_BY_WEBGIS if f not in present]
print("\nSemua file wajib kontrak WebGIS ADA." if not missing else f"\nMASIH KURANG: {missing}")

extra_landcover = sorted(y for y in available_years if y not in (T1_YEAR, T2_YEAR))
if extra_landcover:
    print(f"Bonus landcover tahun {extra_landcover} ikut tersimpan -- backend/frontend perlu "
          f"diperluas (VALID_YEARS di webgis/backend/app/core/config.py, "
          f"AVAILABLE_LANDCOVER_YEARS di frontend) kalau mau ditampilkan di slider tahun.")
if model_metrics is None:
    print("INGAT: metrics.json BELUM dibuat (METRICS_SRC_JSON=None) -- isi nanti & jalankan "
          "ulang Bagian 5 saja begitu file metrik training tersedia.")
if PROVINCE_BOUNDARY_GEOJSON is None:
    print("INGAT: per_province di statistics.json kosong utk data asli -- lihat catatan gap "
          "data batas provinsi di markdown paling atas.")
